# Bloque 1: Configuración e Importación de dependencias

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import os
import csv
import json

In [2]:
VIDEO_PATH = "VideosAnalisis\clip 1 ‐ Hecho con Clipchamp.mp4"
MAPA_PATH = "beachvolleyballcourt.png"
MODEL_PATH = "yolo11n.pt"
MARGIN_PERCENT = 0.10
EXPECTED_PLAYERS = 4

print("✓ Configuración cargada")

model = YOLO(MODEL_PATH)
print("✓ Modelo YOLO cargado")


✓ Configuración cargada
✓ Modelo YOLO cargado


## Definición de funciones auxiliares

In [3]:
def get_points(event, x, y, flags, params):
    """Callback para seleccionar puntos con el mouse."""
    points = params["points"]
    image = params["image"]
    wname = params["wname"]
    max_points = params["max_points"]

    if event == cv2.EVENT_LBUTTONDOWN and len(points) < max_points:
        points.append([x, y])
        cv2.circle(image, (x, y), 6, (0, 0, 255), -1)
        cv2.putText(image, str(len(points)), (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        cv2.imshow(wname, image)

        if len(points) == max_points:
            cv2.waitKey(300)
            cv2.destroyWindow(wname)

def point_in_polygon_with_margin(point, polygon, margin_percent=0.10):
    """Verifica si un punto está dentro de un polígono expandido."""
    center = np.mean(polygon, axis=0)
    expanded_polygon = []
    max_y = np.max(polygon[:, 1])
    
    for pt in polygon:
        if abs(pt[1] - max_y) < 5:
            direction = pt - center
            direction[1] = min(0, direction[1])
            expanded_pt = pt + direction * margin_percent
        else:
            direction = pt - center
            expanded_pt = pt + direction * margin_percent
        expanded_polygon.append(expanded_pt)
    
    expanded_polygon = np.array(expanded_polygon, dtype=np.int32)
    result = cv2.pointPolygonTest(expanded_polygon, point, False)
    return result >= 0


def calculate_iou(box1, box2):
    """Calcula Intersection over Union entre dos bounding boxes."""
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    if x2_i < x1_i or y2_i < y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0

def get_track_info(tracking_data, track_id):
    """Obtiene información completa de un track."""
    frames = []
    positions = []
    bboxes = []
    
    for frame_idx, detections in tracking_data.items():
        for det in detections:
            if det[0] == track_id:
                frames.append(frame_idx)
                positions.append((det[5], det[6]))
                bboxes.append((det[1], det[2], det[3], det[4]))
    
    if not frames:
        return None, None, [], []
    
    return min(frames), max(frames), positions, bboxes

def should_merge_ids(tracking_data, id1, id2, max_gap=5, max_distance=150, min_iou=0.3):
    """Determina si dos IDs deberían fusionarse."""
    first1, last1, positions1, bboxes1 = get_track_info(tracking_data, id1)
    first2, last2, positions2, bboxes2 = get_track_info(tracking_data, id2)
    
    if first1 is None or first2 is None:
        return False
    
    # CASO 1: Secuencial
    if last1 < first2:
        gap = first2 - last1
        if gap <= max_gap:
            last_pos1 = positions1[-1]
            first_pos2 = positions2[0]
            distance = np.sqrt((last_pos1[0] - first_pos2[0])**2 + 
                              (last_pos1[1] - first_pos2[1])**2)
            
            last_bbox1 = bboxes1[-1]
            first_bbox2 = bboxes2[0]
            size1 = (last_bbox1[2] - last_bbox1[0]) * (last_bbox1[3] - last_bbox1[1])
            size2 = (first_bbox2[2] - first_bbox2[0]) * (first_bbox2[3] - first_bbox2[1])
            size_ratio = min(size1, size2) / max(size1, size2) if max(size1, size2) > 0 else 0
            
            if distance <= max_distance and size_ratio > 0.5:
                return True
    
    # CASO 2: Solapamiento
    overlap_start = max(first1, first2)
    overlap_end = min(last1, last2)
    
    if overlap_start <= overlap_end:
        overlapping_frames = []
        for frame_idx in range(overlap_start, overlap_end + 1):
            if frame_idx not in tracking_data:
                continue
            
            bbox1, bbox2 = None, None
            pos1, pos2 = None, None
            
            for det in tracking_data[frame_idx]:
                if det[0] == id1:
                    bbox1 = (det[1], det[2], det[3], det[4])
                    pos1 = (det[5], det[6])
                if det[0] == id2:
                    bbox2 = (det[1], det[2], det[3], det[4])
                    pos2 = (det[5], det[6])
            
            if bbox1 and bbox2:
                distance = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
                iou = calculate_iou(bbox1, bbox2)
                overlapping_frames.append((distance, iou))
        
        if overlapping_frames:
            avg_distance = np.mean([d for d, _ in overlapping_frames])
            avg_iou = np.mean([iou for _, iou in overlapping_frames])
            
            if avg_iou >= min_iou or avg_distance <= max_distance * 0.5:
                return True
    
    return False

def merge_track_ids(tracking_data, id_from, id_to):
    """Fusiona id_from en id_to."""
    for frame_idx in tracking_data:
        new_detections = []
        for det in tracking_data[frame_idx]:
            if det[0] == id_from:
                new_det = (id_to,) + det[1:]
                new_detections.append(new_det)
            else:
                new_detections.append(det)
        tracking_data[frame_idx] = new_detections

def count_frames_with_excess(tracking_data, max_expected=4):
    """Cuenta frames con más jugadores del esperado."""
    return sum(1 for dets in tracking_data.values() if len(dets) > max_expected)

print("✓ Funciones auxiliares definidas")


✓ Funciones auxiliares definidas


# Bloque 2: Obtención de POI

In [4]:
video = cv2.VideoCapture(VIDEO_PATH)
if not video.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"✓ Video cargado: {width}x{height}, {fps:.1f} FPS, {total_frames} frames")

ret, first_frame = video.read()
if not ret:
    raise RuntimeError("No se pudo leer el primer frame")

mapa = cv2.imread(MAPA_PATH)
if mapa is None:
    raise FileNotFoundError(f"No se pudo cargar el mapa: {MAPA_PATH}")

print(f"✓ Mapa cargado: {mapa.shape[1]}x{mapa.shape[0]}")
puntos_campo = []
N = 4

imgA = first_frame.copy()
cv2.namedWindow("Selecciona 4 esquinas del campo", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona 4 esquinas del campo", 1200, 800)
cv2.imshow("Selecciona 4 esquinas del campo", imgA)

cv2.setMouseCallback(
    "Selecciona 4 esquinas del campo",
    get_points,
    {"points": puntos_campo, "image": imgA, 
     "wname": "Selecciona 4 esquinas del campo", "max_points": N}
)

print("Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)")
cv2.waitKey(0)
cv2.destroyAllWindows()

puntos_campo = np.array(puntos_campo, dtype=np.float32)
print(f"✓ {len(puntos_campo)} puntos seleccionados")

# --- Selección de puntos correspondientes en el MAPA ---
puntos_mapa = []

imgB = mapa.copy()
cv2.namedWindow("Selecciona las MISMAS 4 esquinas en el mapa", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona las MISMAS 4 esquinas en el mapa", 800, 600)
cv2.imshow("Selecciona las MISMAS 4 esquinas en el mapa", imgB)

cv2.setMouseCallback(
    "Selecciona las MISMAS 4 esquinas en el mapa",
    get_points,
    {"points": puntos_mapa, "image": imgB,
     "wname": "Selecciona las MISMAS 4 esquinas en el mapa", "max_points": 4}
)

cv2.waitKey(0)
cv2.destroyAllWindows()

puntos_mapa = np.array(puntos_mapa, dtype=np.float32)

assert len(puntos_campo) == 4 and len(puntos_mapa) == 4, "Error en selección de puntos"

H, status = cv2.findHomography(puntos_campo, puntos_mapa, cv2.RANSAC)

if H is None:
    raise RuntimeError("No se pudo calcular la homografía")

print("✓ Homografía calculada correctamente")


✓ Video cargado: 1920x1080, 30.0 FPS, 481 frames
✓ Mapa cargado: 1536x1024
Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)
✓ 4 puntos seleccionados
✓ Homografía calculada correctamente


# Bloque 3: Detección de Jugadores

In [5]:
video.set(cv2.CAP_PROP_POS_FRAMES, 0)
tracking_data = {}

print(f"\nProcesando {total_frames} frames...")
print("Esto puede tardar unos minutos...\n")

frame_idx = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    results = model.track(frame, persist=True, verbose=False, classes=[0])
    frame_detections = []
    
    for r in results:
        if r.boxes.id is None:
            continue
            
        for box, track_id in zip(r.boxes, r.boxes.id):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = (x1 + x2) // 2
            cy = y2
            
            if point_in_polygon_with_margin((cx, cy), puntos_campo, MARGIN_PERCENT):
                frame_detections.append((
                    int(track_id.item()),
                    x1, y1, x2, y2,
                    cx, cy
                ))
    
    tracking_data[frame_idx] = frame_detections
    
    if frame_idx % 50 == 0:
        progress = (frame_idx / total_frames) * 100
        print(f"  Frame {frame_idx}/{total_frames} ({progress:.1f}%) - {len(frame_detections)} jugadores detectados")
    
    frame_idx += 1

print(f"\n✓ Tracking completado: {len(tracking_data)} frames procesados")

# Estadísticas iniciales
all_track_ids = set()
for dets in tracking_data.values():
    for det in dets:
        all_track_ids.add(det[0])

total_detections = sum(len(dets) for dets in tracking_data.values())
print(f"  Total detecciones: {total_detections}")
print(f"  IDs únicos detectados: {len(all_track_ids)}")
print(f"  IDs: {sorted(all_track_ids)}")



Procesando 481 frames...
Esto puede tardar unos minutos...

  Frame 0/481 (0.0%) - 3 jugadores detectados
  Frame 50/481 (10.4%) - 4 jugadores detectados
  Frame 100/481 (20.8%) - 4 jugadores detectados
  Frame 150/481 (31.2%) - 4 jugadores detectados
  Frame 200/481 (41.6%) - 4 jugadores detectados
  Frame 250/481 (52.0%) - 4 jugadores detectados
  Frame 300/481 (62.4%) - 4 jugadores detectados
  Frame 350/481 (72.8%) - 5 jugadores detectados
  Frame 400/481 (83.2%) - 4 jugadores detectados
  Frame 450/481 (93.6%) - 5 jugadores detectados

✓ Tracking completado: 481 frames procesados
  Total detecciones: 1986
  IDs únicos detectados: 33
  IDs: [2, 3, 4, 5, 21, 22, 24, 26, 27, 36, 43, 53, 60, 67, 68, 69, 71, 72, 78, 79, 81, 89, 99, 103, 106, 108, 129, 130, 133, 137, 143, 149, 160]


## Corrección

Elimininación de duplicados

In [6]:
print("\n" + "=" * 70)
print("CORRECCIÓN AVANZADA DE IDS DUPLICADOS")
print("=" * 70)

all_ids = sorted(all_track_ids)
print(f"\n📊 Estado inicial:")
print(f"   IDs detectados: {all_ids}")
print(f"   Total IDs: {len(all_ids)}")
print(f"   Frames con >4 jugadores: {count_frames_with_excess(tracking_data)}")

merge_map = {id: id for id in all_ids}
total_merges = 0

params = [
    (5, 100, 0.4, "Muy estricto - gaps pequeños"),
    (10, 150, 0.3, "Estricto - gaps medianos"),
    (15, 200, 0.25, "Moderado - gaps más largos"),
    (20, 250, 0.2, "Permisivo - oclusiones largas"),
    (30, 350, 0.15, "Muy permisivo - último intento"),
]

for iteration, (max_gap, max_distance, min_iou, description) in enumerate(params, 1):
    print(f"\n{'─' * 70}")
    print(f"ITERACIÓN {iteration}: {description}")
    print(f"   Parámetros: gap≤{max_gap}f, dist≤{max_distance}px, IoU≥{min_iou}")
    print(f"{'─' * 70}")
    
    current_ids = set()
    for dets in tracking_data.values():
        for det in dets:
            current_ids.add(det[0])
    current_ids = sorted(current_ids)
    
    frames_excess = count_frames_with_excess(tracking_data)
    
    print(f"   IDs actuales: {current_ids} ({len(current_ids)} IDs)")
    print(f"   Frames problemáticos: {frames_excess}")
    
    if len(current_ids) <= EXPECTED_PLAYERS and frames_excess < 10:
        print(f"   ✅ ¡Objetivo alcanzado!")
        break
    
    merge_candidates = []
    
    for i, id1 in enumerate(current_ids):
        for id2 in current_ids[i+1:]:
            if should_merge_ids(tracking_data, id1, id2, max_gap, max_distance, min_iou):
                impact = 0
                for frame_idx, dets in tracking_data.items():
                    ids_in_frame = [d[0] for d in dets]
                    if id1 in ids_in_frame and id2 in ids_in_frame:
                        impact += 1
                
                merge_candidates.append((id1, id2, impact))
    
    merge_candidates.sort(key=lambda x: x[2], reverse=True)
    print(f"   Candidatos encontrados: {len(merge_candidates)}")
    
    if not merge_candidates:
        print(f"   ⚠️  No se encontraron fusiones posibles")
        continue
    
    merges_in_iteration = 0
    for id1, id2, impact in merge_candidates:
        current_id1 = merge_map.get(id1, id1)
        current_id2 = merge_map.get(id2, id2)
        
        if current_id1 == current_id2:
            continue
        
        merge_from = max(current_id1, current_id2)
        merge_to = min(current_id1, current_id2)
        
        print(f"      → Fusionando ID {merge_from} → ID {merge_to} (resuelve {impact} frames)")
        
        merge_track_ids(tracking_data, merge_from, merge_to)
        
        for key in merge_map:
            if merge_map[key] == merge_from:
                merge_map[key] = merge_to
        merge_map[merge_from] = merge_to
        
        merges_in_iteration += 1
        total_merges += 1
    
    print(f"   ✓ Fusiones realizadas: {merges_in_iteration}")

print(f"\n{'=' * 70}")
print("✅ CORRECCIÓN COMPLETADA")
print(f"{'=' * 70}")

final_ids = set()
for dets in tracking_data.values():
    for det in dets:
        final_ids.add(det[0])
final_ids = sorted(final_ids)

print(f"\n📊 Resumen de cambios:")
print(f"   IDs originales: {len(all_ids)} → IDs finales: {len(final_ids)}")
print(f"   Total de fusiones: {total_merges}")
print(f"   IDs finales: {final_ids}")

print(f"\n📈 Distribución de jugadores por frame:")
distribution = defaultdict(int)
for dets in tracking_data.values():
    distribution[len(dets)] += 1

for num_players in sorted(distribution.keys()):
    count = distribution[num_players]
    percentage = (count / len(tracking_data)) * 100
    bar = "█" * int(percentage / 2)
    marker = "✓" if num_players == EXPECTED_PLAYERS else "⚠" if num_players > EXPECTED_PLAYERS else "!"
    print(f"   {marker} {num_players} jugadores: {count:4d} frames ({percentage:5.1f}%) {bar}")

frames_exact = distribution.get(EXPECTED_PLAYERS, 0)
frames_over = sum(count for num, count in distribution.items() if num > EXPECTED_PLAYERS)
frames_under = sum(count for num, count in distribution.items() if num < EXPECTED_PLAYERS)

print(f"\n🎯 Calidad del tracking:")
print(f"   Frames perfectos (4 jugadores): {frames_exact} ({frames_exact/len(tracking_data)*100:.1f}%)")
print(f"   Frames con exceso (>4): {frames_over} ({frames_over/len(tracking_data)*100:.1f}%)")
print(f"   Frames con déficit (<4): {frames_under} ({frames_under/len(tracking_data)*100:.1f}%)")

if len(final_ids) == EXPECTED_PLAYERS:
    print(f"\n🎉 ¡PERFECTO! Exactamente {EXPECTED_PLAYERS} jugadores detectados")
elif len(final_ids) < EXPECTED_PLAYERS:
    print(f"\n⚠️  Solo {len(final_ids)} jugadores detectados (esperados: {EXPECTED_PLAYERS})")
else:
    print(f"\n⚠️  {len(final_ids)} jugadores detectados (esperados: {EXPECTED_PLAYERS})")

all_track_ids = final_ids



CORRECCIÓN AVANZADA DE IDS DUPLICADOS

📊 Estado inicial:
   IDs detectados: [2, 3, 4, 5, 21, 22, 24, 26, 27, 36, 43, 53, 60, 67, 68, 69, 71, 72, 78, 79, 81, 89, 99, 103, 106, 108, 129, 130, 133, 137, 143, 149, 160]
   Total IDs: 33
   Frames con >4 jugadores: 138

──────────────────────────────────────────────────────────────────────
ITERACIÓN 1: Muy estricto - gaps pequeños
   Parámetros: gap≤5f, dist≤100px, IoU≥0.4
──────────────────────────────────────────────────────────────────────
   IDs actuales: [2, 3, 4, 5, 21, 22, 24, 26, 27, 36, 43, 53, 60, 67, 68, 69, 71, 72, 78, 79, 81, 89, 99, 103, 106, 108, 129, 130, 133, 137, 143, 149, 160] (33 IDs)
   Frames problemáticos: 138
   Candidatos encontrados: 31
      → Fusionando ID 129 → ID 81 (resuelve 23 frames)
      → Fusionando ID 36 → ID 2 (resuelve 16 frames)
      → Fusionando ID 22 → ID 4 (resuelve 15 frames)
      → Fusionando ID 68 → ID 4 (resuelve 14 frames)
      → Fusionando ID 69 → ID 2 (resuelve 10 frames)
      → Fusionan

In [7]:
print(f"\n{'=' * 70}")
print("ANÁLISIS FINAL DEL TRACKING")
print(f"{'=' * 70}")

track_durations_final = defaultdict(int)

for frame_idx, detections in tracking_data.items():
    for det in detections:
        track_id = det[0]
        track_durations_final[track_id] += 1

print("\nDuración de cada track (en frames):")
for track_id in sorted(track_durations_final.keys()):
    duration = track_durations_final[track_id]
    duration_sec = duration / fps
    percentage = (duration / total_frames) * 100
    print(f"  ID {track_id}: {duration} frames ({duration_sec:.1f}s, {percentage:.1f}% del video)")

print(f"\n📊 Resumen:")
print(f"  • Jugadores únicos: {len(final_ids)}")
print(f"  • Frame más poblado: {max(len(dets) for dets in tracking_data.values())} jugadores")
print(f"  • Frame menos poblado: {min(len(dets) for dets in tracking_data.values())} jugadores")



ANÁLISIS FINAL DEL TRACKING

Duración de cada track (en frames):
  ID 2: 489 frames (16.3s, 101.7% del video)
  ID 3: 493 frames (16.4s, 102.5% del video)
  ID 4: 512 frames (17.1s, 106.4% del video)
  ID 5: 492 frames (16.4s, 102.3% del video)

📊 Resumen:
  • Jugadores únicos: 4
  • Frame más poblado: 7 jugadores
  • Frame menos poblado: 0 jugadores


# Bloque 4: Exportación de datos a CSV

In [8]:
print(f"\n{'=' * 70}")
print("EXPORTANDO DATOS")
print(f"{'=' * 70}\n")

tracking_list = []

for frame_idx, detections in sorted(tracking_data.items()):
    for det in detections:
        track_id, x1, y1, x2, y2, cx, cy = det
        tracking_list.append({
            'frame': frame_idx,
            'track_id': track_id,
            'bbox_x1': x1,
            'bbox_y1': y1,
            'bbox_x2': x2,
            'bbox_y2': y2,
            'center_x': cx,
            'center_y': cy,
            'timestamp_sec': frame_idx / fps
        })

csv_filename = "tracking_data.csv"
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['frame', 'track_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 
                  'center_x', 'center_y', 'timestamp_sec']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(tracking_list)

print(f"✓ Datos exportados a '{csv_filename}'")
print(f"  Total de registros: {len(tracking_list)}")

# Actualización del JSON para incluir puntos del mapa
summary_data = {
    'video_info': {
        'fps': fps,
        'total_frames': total_frames,
        'width': width,
        'height': height,
        'video_path': VIDEO_PATH
    },
    'homografia': {
        'puntos_campo': puntos_campo.tolist(),
        'puntos_mapa': puntos_mapa.tolist(), # <--- Nuevo
        'matrix_H': H.tolist()               # <--- Matriz lista para reuso
    },
    'jugadores_ids': sorted(final_ids),
    'num_jugadores': len(final_ids)
}

json_filename = "tracking_summary.json"
with open(json_filename, 'w', encoding='utf-8') as jsonfile:
    json.dump(summary_data, jsonfile, indent=2)

print(f"✓ Resumen completo exportado a '{json_filename}'")

print(f"\n📁 Archivos generados en: {os.getcwd()}")



EXPORTANDO DATOS

✓ Datos exportados a 'tracking_data.csv'
  Total de registros: 1986
✓ Resumen completo exportado a 'tracking_summary.json'

📁 Archivos generados en: c:\Users\mcash\Documents\4º_GII\Beachvolley-Homography\Beachvolley-Homography


# Representación del resultado

In [9]:
def visualizar_resultado_fluido(video_path, mapa_path, tracking_data, homography_matrix):
    cv2.destroyAllWindows()
    cap = cv2.VideoCapture(video_path)
    mapa_img = cv2.imread(mapa_path)
    
    if not cap.isOpened() or mapa_img is None:
        print("❌ Error: No se pudo cargar el video o el mapa.")
        return

    WINDOW_NAME = "Analisis Beach Volley - Vista Fluida"
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
    
    # Parámetros de estilo y escala
    TARGET_VIDEO_WIDTH = 1000 
    MINIMAP_WIDTH = 550
    COLOR_PASTEL = (240, 207, 137) # El azul glaciar que nos gustó
    
    # --- MEMORIA PARA FLUIDEZ ---
    # Guardará {track_id: (mx, my)}
    last_known_positions = {}

    print("▶️ Reproduciendo con persistencia de movimiento... [Q] para salir.")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        mapa_out = mapa_img.copy()
        ids_en_este_frame = []

        # 1. Procesar detecciones actuales
        if frame_idx in tracking_data:
            for det in tracking_data[frame_idx]:
                tid, x1, y1, x2, y2, cx, cy = det
                ids_en_este_frame.append(tid)
                
                # Dibujo en Video
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(frame, (cx, cy), 6, (0, 255, 0), -1)
                cv2.putText(frame, f"ID {tid}", (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                # Proyección y actualización de memoria
                pt = np.array([[cx, cy]], dtype=np.float32).reshape(-1, 1, 2)
                punto_proyectado = cv2.perspectiveTransform(pt, homography_matrix).reshape(-1, 2)[0]
                mx, my = int(punto_proyectado[0]), int(punto_proyectado[1])
                last_known_positions[tid] = (mx, my)

        # 2. Dibujar en el Mapa (Actuales + Persistentes)
        for tid, (mx, my) in last_known_positions.items():
            # Si el ID no está en este frame, lo dibujamos un poco más pequeño/transparente
            es_fantasma = tid not in ids_en_este_frame
            radio = 25
            color = COLOR_PASTEL if not es_fantasma else (200, 200, 200) # Gris si se perdió
            
            # Dibujo estético del punto
            cv2.circle(mapa_out, (mx, my), radio + 4, (255, 255, 255), -1) # Borde blanco
            cv2.circle(mapa_out, (mx, my), radio, color, -1)
            cv2.putText(mapa_out, str(tid), (mx - 10, my + 8), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2, cv2.LINE_AA)

        # 3. Composición y Redimensionado
        h_v, w_v = frame.shape[:2]
        scale_v = TARGET_VIDEO_WIDTH / w_v
        frame_res = cv2.resize(frame, (TARGET_VIDEO_WIDTH, int(h_v * scale_v)))
        
        h_m, w_m = mapa_out.shape[:2]
        scale_m = MINIMAP_WIDTH / w_m
        mapa_res = cv2.resize(mapa_out, (MINIMAP_WIDTH, int(h_m * scale_m)))
        
        height_final = max(frame_res.shape[0], mapa_res.shape[0])
        
        def add_padding(img, target_h):
            h, w = img.shape[:2]
            return cv2.copyMakeBorder(img, (target_h-h)//2, target_h-h-(target_h-h)//2, 0, 0, 
                                     cv2.BORDER_CONSTANT, value=(20, 20, 20))

        combined_view = cv2.hconcat([add_padding(frame_res, height_final), 
                                     add_padding(mapa_res, height_final)])
        
        cv2.putText(combined_view, f"Frame: {frame_idx}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        cv2.imshow(WINDOW_NAME, combined_view)
        if cv2.waitKey(25) & 0xFF in [27, ord('q')]: break
        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Visualización fluida finalizada.")

# Ejecutar la celda
visualizar_resultado_fluido(VIDEO_PATH, MAPA_PATH, tracking_data, H)

▶️ Reproduciendo con persistencia de movimiento... [Q] para salir.
✅ Visualización fluida finalizada.


In [5]:
import cv2
import numpy as np
import math

# --- CONFIGURACIÓN ---
VIDEO_PATH = r"VideosAnalisis\clip 6 ‐ Hecho con Clipchamp.mp4"
NUM_FRAMES = 150

# ============================================================
# 1. GENERACIÓN DE LA IMAGEN MEDIA (REDUCCIÓN DE RUIDO)
# ============================================================
def generar_imagen_media(video_path, num_frames):
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), "No se pudo abrir el vídeo"
    acc = None
    count = 0
    while count < num_frames:
        ret, frame = cap.read()
        if not ret: break
        frame_f = frame.astype(np.float32)
        acc = frame_f if acc is None else acc + frame_f
        count += 1
    cap.release()
    if count == 0: raise ValueError("No se pudieron leer frames del vídeo.")
    avg = (acc / count).astype(np.uint8)
    return avg

avg = generar_imagen_media(VIDEO_PATH, NUM_FRAMES)

# ============================================================
# 2. SEGMENTACIÓN DE LA ARENA EN ESPACIO HSV
# ============================================================
hsv = cv2.cvtColor(avg, cv2.COLOR_BGR2HSV)
lower_sand = np.array([10, 25, 135])
upper_sand = np.array([35, 140, 255])
mask_sand = cv2.inRange(hsv, lower_sand, upper_sand)

# ============================================================
# 3. LIMPIEZA MORFOLÓGICA Y ROI ESPACIAL
# ============================================================
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
mask_clean = cv2.morphologyEx(mask_sand, cv2.MORPH_CLOSE, kernel)
mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN, kernel)

h_img, w_img = mask_clean.shape
roi = np.zeros_like(mask_clean)
# Definición de márgenes (ejemplo: 5% en todos los lados)
margen_sup = 0.60
margen_inf = 0.10
margen_izq = 0.05
margen_der = 0.05

roi = np.zeros_like(mask_clean)

# Aplicar el rebanado: [y_inicio : y_fin, x_inicio : x_fin]
roi[int(h_img * margen_sup) : int(h_img * (1 - margen_inf)), 
    int(w_img * margen_izq) : int(w_img * (1 - margen_der))] = 255
mask_roi = cv2.bitwise_and(mask_clean, roi)

kernel_horizontal = cv2.getStructuringElement(cv2.MORPH_RECT, (80, 15))
mask_joined = cv2.morphologyEx(mask_roi, cv2.MORPH_CLOSE, kernel_horizontal)

# ============================================================
# CÁLCULO DEL CENTRO DEL CAMPO (SOLO PARA REFLEJO VERTICAL)
# ============================================================
ys, xs = np.where(mask_joined > 0)
if len(xs) == 0:
    raise ValueError("No se pudo calcular el centro del campo")
cx_field = int(xs.mean())

kernel_small = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
mask_joined = cv2.morphologyEx(mask_joined, cv2.MORPH_OPEN, kernel_small)

# ============================================================
# 4. DETECCIÓN DEL CONTORNO Y CUADRILÁTERO
# ============================================================
contours, _ = cv2.findContours(mask_joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
main_cnt = max(contours, key=cv2.contourArea) if contours else None
if main_cnt is None: exit()

epsilon = 0.01 * cv2.arcLength(main_cnt, True)
approx = cv2.approxPolyDP(main_cnt, epsilon, True)
if len(approx) != 4:
    rect = cv2.minAreaRect(main_cnt)
    quad = cv2.boxPoints(rect).astype(int)
else:
    quad = approx.reshape(4, 2)

# ============================================================
# 5. BORDES Y PREPARACIÓN DE HOUGH
# ============================================================
edges_source = cv2.GaussianBlur(mask_sand.copy(), (15, 11), 0)
edges_raw = cv2.Canny(edges_source, 50, 150)
mask_field = np.zeros_like(edges_raw)
cv2.fillPoly(mask_field, [quad.astype(int)], 255)
edges_in_field = cv2.bitwise_and(edges_raw, mask_field)

# ============================================================
# 6. FILTRADO DE LÍNEAS (CORRECCIÓN VERTICALES)
# ============================================================

# --- Horizontales (HoughLinesP: Usa ángulo visual) ---
lines_h = cv2.HoughLinesP(edges_in_field, 1, np.pi/180, 75, minLineLength=100, maxLineGap=20)
margin_h = np.deg2rad(7.5)
lineas_horizontal = []
if lines_h is not None:
    for x1, y1, x2, y2 in lines_h.reshape(-1,4):
        angle = math.atan2(y2 - y1, x2 - x1) % np.pi
        if abs(angle) < margin_h or abs(angle - np.pi) < margin_h:
            lineas_horizontal.append((x1, y1, x2, y2))

# --- Verticales (HoughLines: Usa ángulo normal theta) ---
lines_v = cv2.HoughLines(edges_in_field, 1, np.pi/180, threshold=60)
# margin_v = 45 grados según tu petición
margin_v = np.deg2rad(55) 
lineas_vertical = []

if lines_v is not None:
    for l in lines_v:
        rho, theta = l[0]
        # CORRECCIÓN: Para líneas verticales, theta debe estar cerca de 0 o PI
        if abs(theta) < margin_v or abs(theta - np.pi) < margin_v:
            a, b = np.cos(theta), np.sin(theta)
            x0, y0 = a*rho, b*rho
            x1, y1 = int(x0 + 2000*(-b)), int(y0 + 2000*a)
            x2, y2 = int(x0 - 2000*(-b)), int(y0 - 2000*a)
            lineas_vertical.append((x1, y1, x2, y2))

# --- Funciones de dibujo y unificación ---
def unificar_segmentos(segmentos, tol_rho=15, tol_theta=0.1):
    unificadas = []
    for seg in segmentos:
        x1, y1, x2, y2 = seg
        angle = math.atan2(y2 - y1, x2 - x1) % np.pi
        rho = (x1*y2 - x2*y1) / (np.hypot(x2-x1, y2-y1) + 1e-6)
        agregada = False
        for i, (urho, utheta, useg) in enumerate(unificadas):
            if abs(rho - urho) < tol_rho and abs(angle - utheta) < tol_theta:
                new_seg = (int((x1+useg[0])/2), int((y1+useg[1])/2), int((x2+useg[2])/2), int((y2+useg[3])/2))
                unificadas[i] = ((rho+urho)/2, (angle+utheta)/2, new_seg)
                agregada = True
                break
        if not agregada: unificadas.append((rho, angle, (x1, y1, x2, y2)))
    return [seg for _,_,seg in unificadas]

lineas_unificadas_h = unificar_segmentos(lineas_horizontal, tol_rho=5, tol_theta=0.05)
lineas_unificadas_v = unificar_segmentos(lineas_vertical, tol_rho=15, tol_theta=0.15)

def dibujar_lineas_extendidas(img, segmentos, color, ancho=2):
    h, w = img.shape[:2]
    for x1, y1, x2, y2 in segmentos:
        if x2 - x1 == 0:
            cv2.line(img, (x1,0), (x1,h), color, ancho)
        else:
            m = (y2 - y1)/(x2 - x1)
            b = y1 - m*x1
            cv2.line(img, (0, int(b)), (w, int(m*w + b)), color, ancho)
    return img

# --- Visualización Final ---
debug_proyeccion = avg.copy()
dibujar_lineas_extendidas(debug_proyeccion, lineas_unificadas_h, (255, 0, 255)) # Rosa
dibujar_lineas_extendidas(debug_proyeccion, lineas_unificadas_v, (0, 255, 0))   # Verde

cv2.imshow("Debug - Horizontales Rosa / Verticales Verde", debug_proyeccion)
cv2.waitKey(0)

# ============================================================
# 7. SELECCIÓN DE ESQUINAS POR PROXIMIDAD AL CENTRO (POR CUADRANTE)
# ============================================================

# --- 1. Definición de la función de intersección (necesaria aquí) ---
def obtener_interseccion(l1, l2):
    x1, y1, x2, y2 = l1
    x3, y3, x4, y4 = l2
    den = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(den) < 1e-6: return None
    px = ((x1*y2 - y1*x2)*(x3 - x4) - (x1 - x2)*(x3*y4 - y3*x4)) / den
    py = ((x1*y2 - y1*x2)*(y3 - y4) - (y1 - y2)*(x3*y4 - y3*x4)) / den
    return int(px), int(py)

# --- 2. Filtrar verticales fuera de la zona central (Red) ---
margen_central = w_img * 0.33
v_filtradas = [l for l in lineas_unificadas_v
               if ((l[0]+l[2])/2 < (w_img/2 - margen_central/2)) or
                  ((l[0]+l[2])/2 > (w_img/2 + margen_central/2))]

# --- 3. Calcular TODAS las intersecciones posibles ---
intersecciones = []
for lh in lineas_unificadas_h:
    for lv in v_filtradas:
        pt = obtener_interseccion(lh, lv)
        if pt:
            x, y = pt
            # Filtro básico: que el punto esté dentro del margen de la imagen
            if -100 <= x <= w_img + 100 and -100 <= y <= h_img + 100:
                intersecciones.append(pt)

if len(intersecciones) < 4:
    raise ValueError("No se detectaron suficientes intersecciones.")

# --- 4. Definir centro de referencia y dividir por cuadrantes ---
# cx_field ya viene definido de la sección anterior (media de la máscara)
cy_field = int(h_img * 0.75) 
centro_ref = (cx_field, cy_field)

candidatas_tl, candidatas_tr, candidatas_bl, candidatas_br = [], [], [], []

for pt in intersecciones:
    px, py = pt
    if px < cx_field and py < cy_field:
        candidatas_tl.append(pt)
    elif px >= cx_field and py < cy_field:
        candidatas_tr.append(pt)
    elif px < cx_field and py >= cy_field:
        candidatas_bl.append(pt)
    elif px >= cx_field and py >= cy_field:
        candidatas_br.append(pt)

# --- 5. Selección de la más cercana al centro en cada grupo ---
def obtener_mas_cercano(lista, ref):
    return min(lista, key=lambda p: math.dist(p, ref)) if lista else None

top_left     = obtener_mas_cercano(candidatas_tl, centro_ref)
top_right    = obtener_mas_cercano(candidatas_tr, centro_ref)
bottom_left  = obtener_mas_cercano(candidatas_bl, centro_ref)
bottom_right = obtener_mas_cercano(candidatas_br, centro_ref)

# --- 6. Reflejo de seguridad (Fallback) ---
if not top_left and top_right: top_left = (2*cx_field - top_right[0], top_right[1])
if not top_right and top_left: top_right = (2*cx_field - top_left[0], top_left[1])
if not bottom_left and bottom_right: bottom_left = (2*cx_field - bottom_right[0], bottom_right[1])
if not bottom_right and bottom_left: bottom_right = (2*cx_field - bottom_left[0], bottom_left[1])

esquinas_cuadrilatero = [top_left, top_right, bottom_right, bottom_left]

# Validar que no haya None
if any(v is None for v in esquinas_cuadrilatero):
    print("Error: No se pudieron determinar las 4 esquinas.")
else:
    # Dibujar resultados
    resultado_final = avg.copy()
    for i, pt in enumerate(esquinas_cuadrilatero):
        cv2.circle(resultado_final, pt, 8, (0, 0, 255), -1)
        cv2.putText(resultado_final, f"E{i}", (pt[0], pt[1]-10), 0, 0.6, (0,255,0), 2)
    
    cv2.polylines(resultado_final, [np.array(esquinas_cuadrilatero)], True, (0, 255, 0), 2)
    cv2.imshow("Esquinas por proximidad al centro", resultado_final)
    cv2.waitKey(0)
    cv2.destroyAllWindows()